# Snowflake Cortex AI: Search · Analyst · Agents
### Hands-on scenario notebook + interview prep

**Database:** `SF_SCENARIO_BASED_LEARNING` → new schema **`CORTEX_AI`** (your other schemas are untouched)
**Warehouse:** `COMPUTE_WH`

| Component | Data | Answers |
|---|---|---|
| **Cortex Search** | Unstructured text | "What's our refund policy for defective items?" |
| **Cortex Analyst** | Structured tables (via a semantic view) | "Revenue by region last quarter?" |
| **Cortex Agent** | Both, orchestrated | "Which region had most returns, and why?" |

**Memory hook:** *Search finds text, Analyst writes SQL, and an Agent decides which to use and combines the results.*

**Scenario: "ShopSmart" e-commerce.** Orders, customers and products are structured data for Analyst. Support tickets and policy docs are unstructured data for Search. One Agent covers both.

Run the cells top to bottom.

## 1. Setup

In [ ]:
%%sql -r sql_01
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r sql_02
CREATE SCHEMA IF NOT EXISTS SF_SCENARIO_BASED_LEARNING.CORTEX_AI COMMENT = 'Cortex Search / Analyst / Agents learning'

In [ ]:
%%sql -r sql_03
USE SCHEMA SF_SCENARIO_BASED_LEARNING.CORTEX_AI

In [ ]:
%%sql -r sql_04
USE WAREHOUSE COMPUTE_WH

Cortex needs the `SNOWFLAKE.CORTEX_USER` database role. ACCOUNTADMIN usually already has it.

In [ ]:
%%sql -r sql_05
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE ACCOUNTADMIN

If a model is unavailable in your region, uncomment and run the next cell (it enables cross-region inference).

In [ ]:
%%sql -r sql_06
-- ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
SELECT 'optional step' AS note

## 2. Sample data
### 2.1 Structured tables (used by Analyst)

In [ ]:
%%sql -r sql_07
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.CUSTOMERS (
  CUSTOMER_ID INT, CUSTOMER_NAME STRING, REGION STRING, SEGMENT STRING, SIGNUP_DATE DATE)

In [ ]:
%%sql -r sql_08
INSERT INTO SF_SCENARIO_BASED_LEARNING.CORTEX_AI.CUSTOMERS VALUES
 (1,'Asha Rao','South','Consumer','2024-01-10'),
 (2,'John Smith','North','Corporate','2023-11-05'),
 (3,'Priya Nair','South','Corporate','2024-03-22'),
 (4,'Mike Chen','West','Consumer','2024-02-14'),
 (5,'Sara Ali','East','Home Office','2023-09-30'),
 (6,'Ravi Kumar','North','Consumer','2024-05-01')

In [ ]:
%%sql -r sql_09
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.PRODUCTS (
  PRODUCT_ID INT, PRODUCT_NAME STRING, CATEGORY STRING, UNIT_PRICE NUMBER(10,2))

In [ ]:
%%sql -r sql_10
INSERT INTO SF_SCENARIO_BASED_LEARNING.CORTEX_AI.PRODUCTS VALUES
 (101,'Wireless Earbuds','Electronics',59.99),
 (102,'Smart Watch','Electronics',199.00),
 (103,'Running Shoes','Apparel',89.50),
 (104,'Coffee Maker','Home',120.00),
 (105,'Yoga Mat','Fitness',25.00)

In [ ]:
%%sql -r sql_11
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.ORDERS (
  ORDER_ID INT, CUSTOMER_ID INT, PRODUCT_ID INT, ORDER_DATE DATE,
  QUANTITY INT, AMOUNT NUMBER(10,2), STATUS STRING)

In [ ]:
%%sql -r sql_12
INSERT INTO SF_SCENARIO_BASED_LEARNING.CORTEX_AI.ORDERS VALUES
 (1001,1,101,'2026-08-03',2,119.98,'DELIVERED'),
 (1002,2,102,'2026-08-15',1,199.00,'RETURNED'),
 (1003,3,104,'2026-09-02',1,120.00,'DELIVERED'),
 (1004,4,103,'2026-09-10',2,179.00,'DELIVERED'),
 (1005,5,105,'2026-09-21',4,100.00,'CANCELLED'),
 (1006,6,102,'2026-10-05',1,199.00,'RETURNED'),
 (1007,1,104,'2026-10-12',1,120.00,'DELIVERED'),
 (1008,2,101,'2026-10-20',3,179.97,'DELIVERED'),
 (1009,3,102,'2026-10-28',1,199.00,'RETURNED'),
 (1010,4,105,'2026-11-02',2,50.00,'DELIVERED')

### 2.2 Unstructured tables (used by Search)

In [ ]:
%%sql -r sql_13
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SUPPORT_TICKETS (
  TICKET_ID INT, CUSTOMER_ID INT, PRODUCT_ID INT, REGION STRING, CREATED_AT DATE, TICKET_TEXT STRING)

In [ ]:
%%sql -r sql_14
INSERT INTO SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SUPPORT_TICKETS VALUES
 (1,2,102,'North','2026-08-18','The smart watch battery drains within 4 hours. I want a refund.'),
 (2,6,102,'North','2026-10-07','Smart watch screen flickers and the strap broke after a week.'),
 (3,3,102,'South','2026-10-30','Watch does not sync with my Android phone, returning it.'),
 (4,1,101,'South','2026-08-10','Earbuds are great but the left one is slightly quieter.'),
 (5,4,103,'West','2026-09-15','Shoes arrived late, delivery took 12 days.'),
 (6,5,105,'East','2026-09-22','Cancelled the yoga mat order because shipping cost was too high.')

In [ ]:
%%sql -r sql_15
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_DOCS (DOC_ID INT, TITLE STRING, CONTENT STRING)

In [ ]:
%%sql -r sql_16
INSERT INTO SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_DOCS VALUES
 (1,'Return Policy','Items can be returned within 30 days of delivery. Electronics must be in original packaging. Damaged or defective items get a full refund including shipping.'),
 (2,'Shipping Policy','Standard shipping takes 3-7 business days. Orders above $100 ship free. Express shipping costs $15.'),
 (3,'Warranty Policy','Electronics carry a 1-year manufacturer warranty covering battery and screen defects. Physical damage is not covered.')

In [ ]:
%%sql -r sql_17
SELECT * FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.ORDERS

## 3. Cortex Search
**Concept:** a managed service over a text column. It combines vector and keyword search (hybrid) with semantic reranking. Snowflake manages the embeddings, the index and the refreshes.
- `ON`: the column that gets searched
- `ATTRIBUTES`: columns you can filter on
- `TARGET_LAG`: how fresh the index stays (works like dynamic tables)
- Main use case: **RAG** (retrieve relevant context, then let an LLM generate a grounded answer)

Creating a service can take a minute or two.

## 3.0 Concepts: keyword vs vector vs hybrid search, RAG, and how Cortex Search works

### Keyword search (also called lexical search)
Keyword search matches the **exact words** in your query against the documents.
- Text is split into words (tokens), and an **inverted index** maps each word to the documents that contain it:
```
"battery" → [ticket 1]
"watch"   → [ticket 1, 2, 3]
"refund"  → [ticket 1]
```
- Results are scored with **BM25**, which ranks a word higher when it's frequent in this document but rare across all documents.

| Good at | Weak at |
|---|---|
| Exact terms: SKUs, error codes, names (`ERR-504`) | Synonyms/meaning: "power problem" ≠ "battery drains" |
| Fast, predictable | Paraphrases, typos |

### Vector search (also called semantic search)
Vector search matches on **meaning** rather than exact words.
- An **embedding model** turns text into a vector (a list of numbers, e.g. 768 of them). Similar meanings end up close together:
```
"battery drains in 4 hours"  → [0.12, -0.83, 0.44, ...]
"wearable power problems"    → [0.10, -0.80, 0.47, ...]  ← close (similar meaning)
"shoes arrived late"         → [-0.65, 0.21, 0.09, ...]  ← far
```
- The query is embedded with the same model, and the **nearest vectors** are returned (usually by cosine similarity, using approximate-nearest-neighbour indexes for speed).

| Good at | Weak at |
|---|---|
| Synonyms, paraphrases, intent | Exact codes/IDs (`ERR-504` vs `ERR-505` look almost the same) |
| Natural-language questions | Can return "related but wrong" results |

Snowflake also exposes embeddings directly: `AI_EMBED(...)` returns a `VECTOR` and `VECTOR_COSINE_SIMILARITY(a, b)` compares two of them. Cortex Search does this for you.

### Hybrid search: what Cortex Search uses
```
Query: "smart watch battery refund"
   ├──► Keyword search ──► candidates (exact matches)
   ├──► Vector search  ──► candidates (meaning matches)
   ▼
 Merge  →  Semantic reranker (reads query + candidate together, scores relevance)
   ▼
 Top-k results (ATTRIBUTES filters applied)
```
**Why hybrid:** vector search catches meaning, keyword search catches exact terms, and the reranker puts the best results on top.

### How Cortex Search works behind the scenes
This follows Snowflake's documentation. The internal index technology isn't published.

**When you run `CREATE CORTEX SEARCH SERVICE ... AS (SELECT ...)`:**
1. **Source query:** your warehouse runs the `SELECT`, which defines the rows that get indexed.
2. **Embedding:** the `ON` column is embedded (e.g. `snowflake-arctic-embed-m-v1.5`). This costs embedding tokens plus warehouse time.
3. **Indexing:** Snowflake builds and manages both the vector index and the keyword index.
4. **Serving:** queries are served by Snowflake-managed compute, not your warehouse. That gives low latency and a separate serving charge per GB indexed per month.
5. **Refresh:** like a dynamic table, changes are checked against `TARGET_LAG` and **only changed rows** are re-embedded (incremental refresh).

**When you query it:** the query is embedded with the same model, keyword and vector search run, `filter` conditions on `ATTRIBUTES` columns are applied, the reranker orders the results, and the top `limit` rows are returned.

**Access:** the index is built with the **owner's rights**, and users need `USAGE` on the service. For row-level restrictions, use filters or separate services.

**Files rather than tables?** Cortex Search indexes rows of text. Parse and chunk the files first (see section 5A).

### RAG (Retrieval Augmented Generation)
RAG gives the LLM **your data at question time**, so it answers from facts rather than from memory.
```
Question
   │ 1. RETRIEVE  – Cortex Search returns the top-k relevant chunks
   ▼
   │ 2. AUGMENT   – put those chunks into the prompt as "context"
   ▼
   │ 3. GENERATE  – LLM (AI_COMPLETE) answers ONLY from that context, with citations
   ▼
Grounded answer
```
**Why RAG:**
- LLMs don't know your private or recent data, and RAG supplies it without retraining.
- Grounded context plus "answer only from context" reduces hallucinations.
- Citations make answers verifiable.
- Access to the data stays under Snowflake governance.

**RAG vs fine-tuning:** RAG adds *knowledge* (fresh, private facts), while fine-tuning changes *behaviour* (style, format, a domain task). Many systems use both.

**RAG in Snowflake:**
- Do it yourself: `SEARCH_PREVIEW` (retrieve) + `AI_COMPLETE` (generate). See section 3.3 (tables) and section 5A (files).
- Managed: a **Cortex Agent** with a `cortex_search` tool does retrieve, augment and generate automatically, with citations (section 5).

**Common RAG problems and fixes:**

| Problem | Fix |
|---|---|
| Wrong chunks retrieved | Better chunk size/overlap, add ATTRIBUTES filters, tune `limit` |
| Hallucinated answer | Strict "context only, else say I don't know" prompt; stronger model |
| Answer misses info split across chunks | Increase overlap or chunk size; retrieve more chunks |
| Stale answers | Lower `TARGET_LAG`; incremental ingestion pipeline |

**Interview one-liners:**
- Vector search uses embeddings plus nearest-neighbour similarity and matches meaning.
- Keyword search uses an inverted index plus BM25 and matches exact terms.
- Cortex Search is a hybrid of both plus a semantic reranker: managed, incremental, served by Snowflake.
- RAG means retrieve → augment → generate, which grounds the LLM in your data.

In [ ]:
%%sql -r sql_18
CREATE OR REPLACE CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.TICKET_SEARCH
  ON TICKET_TEXT
  ATTRIBUTES REGION, PRODUCT_ID
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 hour'
AS (
  SELECT TICKET_ID, TICKET_TEXT, REGION, PRODUCT_ID, CUSTOMER_ID, CREATED_AT
  FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SUPPORT_TICKETS
)

In [ ]:
%%sql -r sql_19
CREATE OR REPLACE CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_SEARCH
  ON CONTENT
  ATTRIBUTES TITLE
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
AS (SELECT DOC_ID, TITLE, CONTENT FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_DOCS)

In [ ]:
%%sql -r sql_20
SHOW CORTEX SEARCH SERVICES IN SCHEMA SF_SCENARIO_BASED_LEARNING.CORTEX_AI

### 3.1 Semantic search
None of the words "wearable" or "power" appear in the tickets, but the battery-drain watch ticket should still rank first.

In [ ]:
%%sql -r sql_21
SELECT r.value:TICKET_ID::INT AS TICKET_ID, r.value:REGION::STRING AS REGION, r.value:TICKET_TEXT::STRING AS TICKET_TEXT
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SF_SCENARIO_BASED_LEARNING.CORTEX_AI.TICKET_SEARCH',
  '{"query":"wearable device power problems","columns":["TICKET_ID","TICKET_TEXT","REGION"],"limit":3}'
))['results'])) r

### 3.2 Search with a filter (REGION = North)
Filter operators: `@eq`, `@contains`, `@gte`, `@lte`, `@and`, `@or`, `@not`.

In [ ]:
%%sql -r sql_22
SELECT r.value:REGION::STRING AS REGION, r.value:TICKET_TEXT::STRING AS TICKET_TEXT
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SF_SCENARIO_BASED_LEARNING.CORTEX_AI.TICKET_SEARCH',
  '{"query":"watch issues","columns":["TICKET_TEXT","REGION"],"filter":{"@eq":{"REGION":"North"}},"limit":5}'
))['results'])) r

### 3.3 RAG: Search + LLM
Retrieve the relevant policies, then have an LLM answer using only that context. Change the model name if it isn't available in your region (e.g. `mistral-large2`, `llama3.1-70b`).

In [ ]:
%%sql -r sql_23
WITH ctx AS (
  SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_SEARCH',
    '{"query":"Can I return a smart watch with a battery defect?","columns":["TITLE","CONTENT"],"limit":3}'
  ))['results'] AS r
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
  'claude-3-5-sonnet',
  'Answer ONLY using this context. If unknown, say so. Cite policy titles.\nContext: ' || r::STRING ||
  '\nQuestion: Can I return a smart watch with a battery defect?'
) AS ANSWER
FROM ctx

### 3.4 Same search from Python (Snowflake Python API)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
session = get_active_session()
svc = Root(session).databases["SF_SCENARIO_BASED_LEARNING"].schemas["CORTEX_AI"].cortex_search_services["TICKET_SEARCH"]
resp = svc.search(query="delivery delays and shipping cost", columns=["TICKET_TEXT", "REGION"], limit=3)
print(resp.to_json())

**Production pattern for PDFs** (reference only, not run here):
stage → `SNOWFLAKE.CORTEX.PARSE_DOCUMENT` → `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER` (chunking) → chunk table → `CREATE CORTEX SEARCH SERVICE ... ON CHUNK`.

**Other real-world uses:** HR policy bot, compliance manuals, maintenance logs ("similar failures"), product catalog search, IT incident/RCA knowledge base.

## 4. Cortex Analyst
**Concept:** turns natural-language questions into SQL. Its accuracy comes from a **semantic view** that defines tables, relationships, facts, dimensions, metrics and synonyms (for example, "clients" means customers and "sales" means revenue).
It runs the SQL with the **caller's role**, so RBAC, masking and row access policies all apply.

In [ ]:
%%sql -r sql_24
CREATE OR REPLACE SEMANTIC VIEW SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV
  TABLES (
    customers AS SF_SCENARIO_BASED_LEARNING.CORTEX_AI.CUSTOMERS PRIMARY KEY (CUSTOMER_ID)
      WITH SYNONYMS = ('clients','buyers') COMMENT = 'Customer master',
    products  AS SF_SCENARIO_BASED_LEARNING.CORTEX_AI.PRODUCTS  PRIMARY KEY (PRODUCT_ID) COMMENT = 'Product catalog',
    orders    AS SF_SCENARIO_BASED_LEARNING.CORTEX_AI.ORDERS    PRIMARY KEY (ORDER_ID)   COMMENT = 'One row per order'
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (CUSTOMER_ID) REFERENCES customers,
    orders_to_products  AS orders (PRODUCT_ID)  REFERENCES products
  )
  FACTS (
    orders.order_amount AS AMOUNT COMMENT = 'Order value in USD',
    orders.order_qty    AS QUANTITY
  )
  DIMENSIONS (
    customers.customer_name AS CUSTOMER_NAME,
    customers.region        AS REGION WITH SYNONYMS = ('area','zone'),
    customers.segment       AS SEGMENT,
    products.product_name   AS PRODUCT_NAME,
    products.category       AS CATEGORY,
    orders.order_date       AS ORDER_DATE,
    orders.order_month      AS DATE_TRUNC('MONTH', ORDER_DATE),
    orders.status           AS STATUS COMMENT = 'DELIVERED, RETURNED or CANCELLED'
  )
  METRICS (
    orders.total_revenue   AS SUM(orders.order_amount) WITH SYNONYMS = ('sales','revenue'),
    orders.order_count     AS COUNT(orders.ORDER_ID),
    orders.return_count    AS COUNT_IF(orders.STATUS = 'RETURNED'),
    orders.avg_order_value AS AVG(orders.order_amount)
  )
  COMMENT = 'ShopSmart sales semantic view'

In [ ]:
%%sql -r sql_25
DESCRIBE SEMANTIC VIEW SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV

### 4.1 Query the semantic view with SQL
The metric definition lives in one place, so everyone gets the same number.

In [ ]:
%%sql -r sql_26
SELECT * FROM SEMANTIC_VIEW(
  SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV
  DIMENSIONS customers.region
  METRICS orders.total_revenue, orders.return_count
) ORDER BY total_revenue DESC

In [ ]:
%%sql -r sql_27
SELECT * FROM SEMANTIC_VIEW(
  SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV
  DIMENSIONS orders.order_month
  METRICS orders.total_revenue, orders.order_count
) ORDER BY order_month

### 4.2 Ask in natural language
Open **Snowsight → AI & ML → Cortex Analyst**, select `SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV`, and try:
- *Which region has the highest return count?* (expected: North, with 2)
- *Total revenue by month*
- *Top 3 products by sales*
- *Revenue from corporate clients in South* (this tests the synonym "clients")

From an application, use the REST endpoint `POST /api/v2/cortex/analyst/message` with the body `{"messages":[...], "semantic_view":"SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV"}`.

**Verified queries (VQRs):** in the same UI, save trusted question→SQL pairs. Analyst reuses them and marks those answers as verified.

## 5. Cortex Agent
**Concept:** the orchestrator. For each question it **plans** the work, **uses tools** (Analyst, Search, custom procedures), **reflects** on the results, and **responds** with a combined answer that includes citations and SQL.

In [ ]:
%%sql -r sql_28
CREATE OR REPLACE AGENT SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SHOPSMART_AGENT
  COMMENT = 'Sales + support assistant'
  PROFILE = '{"display_name": "ShopSmart Assistant"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

instructions:
  response: "Be concise. Show numbers in tables. Cite policy titles when used."
  orchestration: >
    Use SalesAnalyst for revenue, order, return or customer metric questions.
    Use TicketSearch for customer complaints and feedback.
    Use PolicySearch for return, shipping or warranty rules.
    For mixed questions call SalesAnalyst first, then search, then combine.
  sample_questions:
    - question: "Which region had the most returns and why?"
    - question: "Is a smart watch with battery issues covered by warranty?"

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: SalesAnalyst
      description: "Structured sales data: orders, revenue, returns, customers, products."
  - tool_spec:
      type: cortex_search
      name: TicketSearch
      description: "Free-text customer support tickets and complaints."
  - tool_spec:
      type: cortex_search
      name: PolicySearch
      description: "Company return, shipping and warranty policies."

tool_resources:
  SalesAnalyst:
    semantic_view: SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV
    execution_environment:
      type: warehouse
      warehouse: COMPUTE_WH
  TicketSearch:
    name: SF_SCENARIO_BASED_LEARNING.CORTEX_AI.TICKET_SEARCH
    max_results: 5
    id_column: TICKET_ID
  PolicySearch:
    name: SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_SEARCH
    max_results: 3
    id_column: DOC_ID
    title_column: TITLE
$$

In [ ]:
%%sql -r sql_29
SHOW AGENTS IN SCHEMA SF_SCENARIO_BASED_LEARNING.CORTEX_AI

In [ ]:
%%sql -r sql_30
DESCRIBE AGENT SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SHOPSMART_AGENT

### 5.1 Chat with the agent
Open **Snowsight → AI & ML → Agents → SHOPSMART_AGENT** (or Snowflake Intelligence) and ask:

> *Which region had the most returns and what are customers complaining about?*

What the agent should do:
1. Plan: split the question into a structured part (returns by region) and an unstructured part (complaints).
2. **SalesAnalyst** finds that North has 2 returns, both Smart Watch.
3. **TicketSearch**, filtered to North, finds battery drain, screen flicker and a broken strap.
4. **PolicySearch** finds that the warranty covers battery and screen defects.
5. It combines these into a single answer with citations.

The REST endpoint is `POST /api/v2/databases/SF_SCENARIO_BASED_LEARNING/schemas/CORTEX_AI/agents/SHOPSMART_AGENT:run`, and the response is streamed as server-sent events.

### 5.2 Custom tool: a stored procedure the agent can call to take an action

In [ ]:
%%sql -r sql_31
CREATE OR REPLACE PROCEDURE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.CREATE_REFUND_REQUEST(ORDER_ID INT, REASON STRING)
RETURNS STRING LANGUAGE SQL AS
$$
BEGIN
  RETURN 'Refund request created for order ' || :ORDER_ID || ' (' || :REASON || ')';
END;
$$

In [ ]:
%%sql -r sql_32
CALL SF_SCENARIO_BASED_LEARNING.CORTEX_AI.CREATE_REFUND_REQUEST(1002, 'Battery defect')

To let the agent call it, add it to the agent as a `generic` tool that points at this procedure. The easiest way is **Agents UI → Edit → Tools → Custom tool**.

## 5A. Cortex Search over files in a stage (PDF / DOCX / PPTX)
Cortex Search indexes **rows of text** (the results of a query), not raw files. It also doesn't chunk text for you, because each source row becomes one search unit. So for files the pipeline is:

```
@stage → AI_PARSE_DOCUMENT (text) → SPLIT_TEXT_RECURSIVE_CHARACTER (chunks) → chunk table → CORTEX SEARCH SERVICE
```

**Why chunk?** Embedding models have input limits, one vector for a 50-page PDF blurs its meaning, and smaller chunks give precise matches and shorter LLM prompts. The overlap keeps sentences from being cut off at chunk boundaries.

> **Cost:** `AI_PARSE_DOCUMENT` is billed per page. Check current rates in the Snowflake docs ("AI Functions costs" and the Service Consumption Table) before you parse many files.

### Step 1: Create a stage
The stage must use server-side encryption (`SNOWFLAKE_SSE`) for parsing to work. Its directory table lists the files.

In [ ]:
%%sql -r files_01
CREATE STAGE IF NOT EXISTS SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')

### Step 2: Upload files
In Snowsight, go to **Data → Databases → SF_SCENARIO_BASED_LEARNING → CORTEX_AI → Stages → DOCS_STAGE → + Files**, upload a few PDFs (HR policy, product manual, etc.), then refresh the directory.

In [ ]:
%%sql -r files_02
ALTER STAGE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE REFRESH

In [ ]:
%%sql -r files_03
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED FROM DIRECTORY(@SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE)

### Step 3: Parse one file (try it first)
`LAYOUT` mode keeps tables and headings as Markdown, and `OCR` mode returns plain text (good for scans). This cell parses only the first file, as a cheap test.

In [ ]:
%%sql -r files_04
SELECT RELATIVE_PATH,
       LEFT(AI_PARSE_DOCUMENT(TO_FILE('@SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE', RELATIVE_PATH), {'mode': 'LAYOUT'}):content::STRING, 2000) AS PREVIEW
FROM DIRECTORY(@SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE)
WHERE RELATIVE_PATH ILIKE '%.pdf'
LIMIT 1

### Step 4: Parse all files into a table
Parse each file once and store the text, so re-chunking later doesn't pay for parsing again.

In [ ]:
%%sql -r files_05
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_TEXT AS
SELECT RELATIVE_PATH AS FILE_NAME,
       AI_PARSE_DOCUMENT(TO_FILE('@SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE', RELATIVE_PATH), {'mode': 'LAYOUT'}):content::STRING AS DOC_TEXT
FROM DIRECTORY(@SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOCS_STAGE)
WHERE RELATIVE_PATH ILIKE ANY ('%.pdf', '%.docx', '%.pptx')

### Step 5: Chunk the text
The arguments are `(text, format, chunk_size, overlap)`. Unlike the `AI_*` functions, `SPLIT_TEXT_RECURSIVE_CHARACTER` needs the `SNOWFLAKE.CORTEX.` prefix.

In [ ]:
%%sql -r files_06
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_CHUNKS AS
SELECT d.FILE_NAME,
       f.index         AS CHUNK_NO,
       f.value::STRING AS CHUNK
FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_TEXT d,
     LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(d.DOC_TEXT, 'markdown', 1500, 200)) f
WHERE d.DOC_TEXT IS NOT NULL

In [ ]:
%%sql -r files_07
SELECT FILE_NAME, COUNT(*) AS CHUNKS, AVG(LENGTH(CHUNK))::INT AS AVG_CHARS FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_CHUNKS GROUP BY FILE_NAME

### Step 6: Create the search service
Create it **only after DOC_CHUNKS has rows**. A service built on an empty table returns nothing, even when its status shows ACTIVE.

In [ ]:
%%sql -r files_08
CREATE OR REPLACE CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_SEARCH
  ON CHUNK
  ATTRIBUTES FILE_NAME
  WAREHOUSE = COMPUTE_WH
  TARGET_LAG = '1 day'
AS (SELECT CHUNK, FILE_NAME, CHUNK_NO FROM SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_CHUNKS)

Check that the index has data. In the output, `source_data_num_rows` should be greater than 0.

In [ ]:
%%sql -r files_09
DESCRIBE CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_SEARCH

### Step 7: Search the documents and answer with citations (RAG)
Change the question to fit your files.

In [ ]:
%%sql -r files_10
SELECT r.value:FILE_NAME::STRING AS FILE_NAME, r.value:CHUNK_NO::INT AS CHUNK_NO, LEFT(r.value:CHUNK::STRING, 300) AS SNIPPET
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_SEARCH',
  '{"query":"what is the leave policy for new joiners","columns":["FILE_NAME","CHUNK_NO","CHUNK"],"limit":3}'
))['results'])) r

In [ ]:
%%sql -r files_11
WITH ctx AS (
  SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'SF_SCENARIO_BASED_LEARNING.CORTEX_AI.DOC_SEARCH',
    '{"query":"what is the leave policy for new joiners","columns":["FILE_NAME","CHUNK"],"limit":3}'
  ))['results'] AS r
)
SELECT AI_COMPLETE(
  'claude-3-5-sonnet',
  'Answer ONLY from this context and cite FILE_NAME. If not found, say so.\nContext: ' || r::STRING ||
  '\nQuestion: What is the leave policy for new joiners?'
) AS ANSWER
FROM ctx

### Step 8: Keep it fresh as new files arrive (production pattern)
The tables above are one-time snapshots, so files uploaded later won't be searchable. In production:
1. **Stream on the stage** (`CREATE STREAM ... ON STAGE DOCS_STAGE`) to detect newly added files.
2. **Task** (scheduled) that inserts new files from the stream into a file-log table.
3. **Dynamic tables with `REFRESH_MODE = INCREMENTAL`** for the parse and chunk steps, so each file is parsed once.
4. The **search service** re-indexes changed rows within its `TARGET_LAG`.

You can also add the `DOC_SEARCH` service as another `cortex_search` tool on `SHOPSMART_AGENT` so the agent can answer from your documents too.

**Interview one-liners:**
- Cortex Search indexes query results, not files: parse, chunk, store in a table, then index.
- A typical chunk is about 1000–1500 characters with 10–20% overlap.
- `AI_PARSE_DOCUMENT` has two modes: `OCR` (text only) and `LAYOUT` (Markdown with tables).
- Check `source_data_num_rows` to confirm the index actually has data.

## 6. Security / RBAC
Grant a role access to everything it needs. The `SHOP_ANALYST` role is created below.

In [ ]:
%%sql -r sql_33
CREATE ROLE IF NOT EXISTS SHOP_ANALYST

In [ ]:
%%sql -r sql_34
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_35
GRANT USAGE ON DATABASE SF_SCENARIO_BASED_LEARNING TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_36
GRANT USAGE ON SCHEMA SF_SCENARIO_BASED_LEARNING.CORTEX_AI TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_37
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_38
GRANT SELECT ON ALL TABLES IN SCHEMA SF_SCENARIO_BASED_LEARNING.CORTEX_AI TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_39
GRANT SELECT ON SEMANTIC VIEW SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SALES_SV TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_40
GRANT USAGE ON CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.TICKET_SEARCH TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_41
GRANT USAGE ON CORTEX SEARCH SERVICE SF_SCENARIO_BASED_LEARNING.CORTEX_AI.POLICY_SEARCH TO ROLE SHOP_ANALYST

In [ ]:
%%sql -r sql_42
GRANT USAGE ON AGENT SF_SCENARIO_BASED_LEARNING.CORTEX_AI.SHOPSMART_AGENT TO ROLE SHOP_ANALYST

## 7. Comparison cheat-sheet
| Aspect | Cortex Search | Cortex Analyst | Cortex Agent |
|---|---|---|---|
| Data | Text | Tables | Both |
| Core tech | Embeddings + keyword + rerank | LLM text-to-SQL + semantic model | LLM planning + tool use |
| Object | CORTEX SEARCH SERVICE | SEMANTIC VIEW | AGENT |
| Output | Ranked chunks | SQL + result | Final answer + citations/SQL |
| Freshness | TARGET_LAG | Live | Depends on tools |
| Cost | Refresh warehouse + serving (GB/month) + embeddings | Per message + warehouse | Tokens + tool costs |

**Decision rule:** an exact number from tables → Analyst. Meaning from text → Search. Both, or multi-step work → Agent.

## 8. Interview Q&A
**1. What is Snowflake Cortex?** Snowflake's managed AI layer: LLM functions (COMPLETE, AI_* functions), Search, Analyst, Agents and Document AI. Everything runs inside Snowflake's governance boundary.

**2. Search vs Analyst?** Search retrieves relevant unstructured text. Analyst generates SQL over structured data and returns an exact answer.

**3. What is RAG?** Retrieve relevant context first, then let an LLM generate an answer grounded in it. This reduces hallucination. Cortex Search is the retrieval layer.

**4. Why hybrid search?** Vector search matches meaning, keyword search matches exact terms (SKUs, error codes), and reranking improves the top results.

**5. What is a semantic view?** A business layer of tables, joins, dimensions, metrics, synonyms and verified queries. It makes the SQL the LLM generates accurate and consistent.

**6. What is a Cortex Agent?** An orchestrator that plans, calls tools, reflects on the results, and answers multi-step questions that span tables and text.

**7. Scenario: different teams report different revenue numbers.** Define the metric once in a semantic view, add verified queries, and expose it through Analyst or an Agent.

**8. Analyst gets "smartwatch" wrong.** The stored value is "Smart Watch". Add synonyms or sample values, link a Cortex Search service to that dimension for value lookup, and add a VQR.

**9. 10,000 PDFs updated daily.** Stage → PARSE_DOCUMENT → chunk → table → Search service with TARGET_LAG = '1 day' and attribute filters → Agent with citations.

**10. "Most returns and why?"** Use an Agent: Analyst answers "most" and Search answers "why".

**11. Regional manager sees only their region.** Put a row access policy on the base tables. Analyst and Agents run as the caller, so it applies. A Search service is built with the owner's rights, so use attribute filters or separate services.

**12. Reduce Search cost.** Index only what you need, set a sensible TARGET_LAG, use a small warehouse, and `ALTER CORTEX SEARCH SERVICE ... SUSPEND` when idle.

**13. Evaluate quality.** Build a golden question set. Measure precision/recall@k for Search and SQL accuracy against VQRs for Analyst. For Agents, use AI observability (groundedness, relevance).

**14. RAG hallucinates.** Improve chunk size and overlap, add filters, use a strict "context only" prompt, return citations, and try a stronger model.

**15. When not to use an Agent?** For single-purpose needs: a dashboard KPI should use a semantic view or SQL, and a document lookup should call Search directly. You get lower latency and cost.

**Rapid-fire:** `SEARCH_PREVIEW` tests search in SQL · `SNOWFLAKE.CORTEX_USER` is the role needed · `/api/v2/cortex/analyst/message` is the Analyst endpoint · `...agents/{name}:run` runs an agent · `ATTRIBUTES` sets filter columns · `TARGET_LAG` sets freshness · Snowflake Intelligence is the business-user chat UI.

## 9. Cleanup (optional)
Uncomment and run when you're finished. It drops only the `CORTEX_AI` schema and the demo role.

In [ ]:
%%sql -r sql_43
-- DROP SCHEMA IF EXISTS SF_SCENARIO_BASED_LEARNING.CORTEX_AI CASCADE;
-- DROP ROLE IF EXISTS SHOP_ANALYST;
SELECT 'cleanup is commented out' AS note